# 02 · Feature pipeline

Builds the feature store described in `docs/decisions.md` (D5–D7), as a notebook first. The code is exported to
`src/feature_pipeline/` once it works.

One run does four things:

1. **Pull** SAKE, WEA and LIK and store every pull with its download date (D5).
2. **Parse** each source into a tidy table, with the date from which every value was available.
3. **Build features**: one row per segment and target quarter, computed as of the release date of the latest SAKE quarter (D7),
   with the year-on-year target (D6). The **backfill** is the same function run for every past release since 2011.
   A **refresh** row set is added for the run date, so new WEA and LIK values reach the feature store between SAKE releases.
4. **Check and store**: leakage checks, then write the feature table as a new dated version to the GCS bucket
   (or a local folder without credentials).

Set `OFFLINE = True` to work from the snapshots of notebook 01 instead of downloading.

## Setup

In [1]:
import hashlib
import io
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# --- Run mode ---
OFFLINE = False  # True: read the dated snapshots from notebook 01 instead of downloading
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())  # repo root
SNAPSHOT_DIR = ROOT / "data/raw/exploration"
SNAPSHOT_DATE = "2026-10-15"

# --- Sources ---
CKAN = "https://ckan.opendata.swiss/api/3/action"
SAKE_DATASET = "erwerbslosenquote-gemass-ilo-nach-geschlecht-nationalitat-und-altersgruppen-durchschnittliche-m"
WEA_URL = "https://scheduler.swissdatas.ch/scheduled/wwa.csv"
LIK_URL = "https://data.snb.ch/api/cube/plkopr/data/csv/en"

# --- Scope (notebook 01, section 3) ---
START_QUARTER = pd.Period("2010Q1", freq="Q")
EXCLUDED_AGE = "65+ Jahre"
N_SEGMENTS = 45

# --- Publication lags, used where the real publication date is unknown (see section 2) ---
SAKE_LAG_DAYS = 49  # quarter end -> BFS release (Q2 2026: 30.06 -> 18.08)
WEA_LAG_DAYS = 24   # week start -> value in the file (observed 24 days on 15.10, 20 days on 25.10)
LIK_LAG_DAYS = 21   # month end -> SNB publication (August 2026 published 21.09)

# --- Storage: GCS bucket if GCS_BUCKET is set in .env (or GitHub Secrets), local folder otherwise ---
load_dotenv(ROOT / ".env")
GCS_BUCKET = os.getenv("GCS_BUCKET")
key_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if key_path and not Path(key_path).is_absolute():
    # .env gives the key path from the repo root, wherever the notebook runs
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(ROOT / key_path)
LOCAL_ROOT = ROOT / "data"
PULL_LOG = "raw/pull_log.csv"
FEATURES_PREFIX = "feature_store/features"

RUN_DATE = SNAPSHOT_DATE if OFFLINE else datetime.now(timezone.utc).strftime("%Y-%m-%d")
print("run date:", RUN_DATE, "| offline" if OFFLINE else "| live")

run date: 2026-10-25 | live


### Storage: local folder or GCS bucket

Everything the pipeline writes goes through a small `store` object with four methods: `write`, `read`, `exists` and
`list`. Files are addressed by a key such as `raw/sake/2026-10-25.csv`, and the key is the same in both places:

| Backend | When | `raw/sake/2026-10-25.csv` ends up in |
|---|---|---|
| `GCSStore` | `GCS_BUCKET` is set and `OFFLINE = False` | `gs://<bucket>/raw/sake/2026-10-25.csv` |
| `LocalStore` | otherwise | `data/raw/sake/2026-10-25.csv` |

So the rest of the notebook doesn't know where it writes, and the same code runs on a laptop without credentials and in
GitHub Actions with them. Offline runs always stay local, so snapshot data never lands in the bucket.

The GCS client finds the service-account key through `GOOGLE_APPLICATION_CREDENTIALS` (set in `.env`, see `.env.example`).

In [2]:
class LocalStore:
    def __init__(self, root: Path):
        self.root = root

    def write(self, key: str, data: bytes) -> None:
        path = self.root / key
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)

    def read(self, key: str) -> bytes:
        return (self.root / key).read_bytes()

    def exists(self, key: str) -> bool:
        return (self.root / key).exists()

    def list(self, prefix: str) -> list[str]:
        """All keys below a prefix, e.g. list('raw/sake')."""
        folder = self.root / prefix
        return sorted(str(p.relative_to(self.root)) for p in folder.rglob("*") if p.is_file()) if folder.exists() else []

    def uri(self, key: str) -> str:
        return str(self.root / key)


class GCSStore:
    def __init__(self, bucket: str):
        from google.cloud import storage  # only needed when a bucket is configured

        self.client = storage.Client()
        self.bucket = self.client.bucket(bucket)

    def write(self, key: str, data: bytes) -> None:
        self.bucket.blob(key).upload_from_string(data)

    def read(self, key: str) -> bytes:
        return self.bucket.blob(key).download_as_bytes()

    def exists(self, key: str) -> bool:
        return self.bucket.blob(key).exists()

    def list(self, prefix: str) -> list[str]:
        return sorted(b.name for b in self.client.list_blobs(self.bucket, prefix=prefix.rstrip("/") + "/"))

    def uri(self, key: str) -> str:
        return f"gs://{self.bucket.name}/{key}"


store = GCSStore(GCS_BUCKET) if GCS_BUCKET and not OFFLINE else LocalStore(LOCAL_ROOT)
print("storage:", store.uri(""))

storage: gs://hslu-hs26-mlops-theodora/


## 1 · Pull the sources (D5)

`http_get` and `ckan` are the retry wrappers from notebook 01: they retry server errors (5xx) and timeouts after 2, 4 and 8 s,
and fail straight away on client errors (4xx).

Each `pull_*` function returns the raw bytes and a small metadata record:

| Field | SAKE | WEA | LIK |
|---|---|---|---|
| `published` | CSV resource `issued` (release date) | not given by SECO | `PublishingDate`, line 2 of the file |
| `latest_period` | latest quarter in the file | latest week in the file | latest month in the file |

For SAKE, the CSV resource is looked up in the CKAN dataset on every run instead of using a fixed URL (A1). If BFS adds a new
release as a separate resource, the one with the latest `issued` date is taken.

In [3]:
def http_get(url: str, retries: int = 4, **params) -> requests.Response:
    """GET with retries on server errors (5xx) and timeouts, waiting 2, 4, 8 s."""
    reason = "no attempt"
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=60)
            if r.status_code < 500:
                r.raise_for_status()
                return r
            reason = f"HTTP {r.status_code}"
        except (requests.ConnectionError, requests.Timeout) as e:
            reason = type(e).__name__
        if attempt < retries - 1:
            wait = 2 ** (attempt + 1)
            print(f"{url}: {reason}, retrying in {wait} s")
            time.sleep(wait)
    raise RuntimeError(f"{url} failed after {retries} attempts ({reason})")


def ckan(action: str, **params) -> dict:
    body = http_get(f"{CKAN}/{action}", **params).json()
    if not body.get("success"):
        raise RuntimeError(body.get("error"))
    return body["result"]


def read_snapshot(name: str, ext: str = ".csv") -> bytes:
    return (SNAPSHOT_DIR / f"{name}_{SNAPSHOT_DATE}{ext}").read_bytes()

In [4]:
def parse_quarter(value):
    """Turn '2024-Q1' (and similar spellings) into a quarterly Period."""
    m = re.search(r"(\d{4})\s*[-/ ]?\s*[QqTtKk]\s*0?([1-4])\b", str(value))
    return pd.Period(year=int(m[1]), quarter=int(m[2]), freq="Q") if m else pd.NaT


def pull_sake() -> tuple[bytes, dict]:
    pkg = json.loads(read_snapshot("sake_ckan", ".json")) if OFFLINE else ckan("package_show", id=SAKE_DATASET)
    csvs = [r for r in pkg["resources"] if "CSV" in (r.get("format") or "").upper()]
    if not csvs:
        raise RuntimeError("SAKE dataset has no CSV resource")
    if len(csvs) > 1:
        print(f"SAKE: {len(csvs)} CSV resources, taking the one with the latest 'issued' date")
    res = max(csvs, key=lambda r: r.get("issued") or "")
    url = res.get("download_url") or res.get("url")
    content = read_snapshot("sake") if OFFLINE else http_get(url).content

    periods = re.findall(rb'"(\d{4}-Q\d)"', content)
    return content, {
        "source": "sake",
        "url": url,
        "resource_id": res["id"],
        "published": (res.get("issued") or "")[:10] or None,
        "latest_period": str(max(parse_quarter(p.decode()) for p in periods)),
    }


def pull_wea() -> tuple[bytes, dict]:
    content = read_snapshot("wea") if OFFLINE else http_get(WEA_URL).content
    dates = pd.read_csv(io.BytesIO(content), usecols=["structure", "date"])
    latest = dates.loc[dates["structure"] == "seco_wwa", "date"].max()
    return content, {"source": "wea", "url": WEA_URL, "resource_id": None, "published": None, "latest_period": latest}


def pull_lik() -> tuple[bytes, dict]:
    content = read_snapshot("lik_snb") if OFFLINE else http_get(LIK_URL).content
    lines = content.decode("utf-8-sig").splitlines()
    published = lines[1].split(";")[1].strip('"')[:10]  # "PublishingDate";"YYYY-MM-DD HH:MM"
    latest = max(line.split(";")[0].strip('"') for line in lines[4:] if line)
    return content, {"source": "lik", "url": LIK_URL, "resource_id": None, "published": published, "latest_period": latest}


pulls = {name: fn() for name, fn in [("sake", pull_sake), ("wea", pull_wea), ("lik", pull_lik)]}
pd.DataFrame([meta | {"bytes": len(content)} for content, meta in pulls.values()]).set_index("source")

,url,resource_id,published,latest_period,bytes
source,,,,,
sake,https://dam-api.bfs.admin.ch/hub/api/dam/asset...,c9b62cc9-84b2-4629-a1be-88b92a4ad00f,2026-08-18,2026Q2,6049609
wea,https://scheduler.swissdatas.ch/scheduled/wwa.csv,NaN,NaN,2026-10-05,62443
lik,https://data.snb.ch/api/cube/plkopr/data/csv/en,NaN,2026-10-21,2026-09,81371


### Store every pull with its download date

Every pull is recorded in `raw/pull_log.csv` with its download date and a SHA-256 hash of the file. The file itself is
only written when its hash is new, as `raw/<source>/<download date>.csv`, so unchanged weekly pulls don't pile up copies.
The `file` column holds the storage key, so the log reads the same locally and in the bucket.

The log answers two questions later on:

- **Which data version did a forecast use?** Every feature version lists the hashes it was built from.
- **Is there a new SAKE release?** A new hash for SAKE is the retraining trigger (D4), used in MS3.

In [5]:
LOG_COLUMNS = ["download_date", "source", "sha256", "is_new", "published", "latest_period", "resource_id", "url", "file"]


def load_pull_log() -> pd.DataFrame:
    if store.exists(PULL_LOG):
        return pd.read_csv(io.BytesIO(store.read(PULL_LOG)), dtype=str)
    return pd.DataFrame(columns=LOG_COLUMNS, dtype=str)


def store_raw(content: bytes, meta: dict, log: pd.DataFrame) -> dict:
    sha = hashlib.sha256(content).hexdigest()
    seen = set(log.loc[log["source"] == meta["source"], "sha256"])
    key = f"raw/{meta['source']}/{RUN_DATE}.csv"
    if sha not in seen:
        store.write(key, content)
    else:
        # Same content as an earlier pull: point to the file that already holds it
        key = log.loc[(log["source"] == meta["source"]) & (log["sha256"] == sha), "file"].iloc[0]
    return {"download_date": RUN_DATE, "sha256": sha, "is_new": str(sha not in seen), "file": key} | meta


pull_log = load_pull_log()
earlier = pull_log[pull_log["download_date"] != RUN_DATE]  # a re-run on the same day compares with earlier days only
this_run = pd.DataFrame([store_raw(content, meta, earlier) for content, meta in pulls.values()])[LOG_COLUMNS]

# Re-running on the same day replaces that day's entries instead of adding duplicates
pull_log = pd.concat([earlier, this_run], ignore_index=True)
store.write(PULL_LOG, pull_log.to_csv(index=False).encode())

new_sake_release = this_run.set_index("source").loc["sake", "is_new"] == "True"
print("new SAKE version (retraining trigger, D4):", new_sake_release)
this_run[["source", "is_new", "published", "latest_period", "file"]]

new SAKE version (retraining trigger, D4): True


,source,is_new,published,latest_period,file
0,sake,True,2026-08-18,2026Q2,raw/sake/2026-10-25.csv
1,wea,True,NaN,2026-10-05,raw/wea/2026-10-25.csv
2,lik,True,2026-10-21,2026-09,raw/lik/2026-10-25.csv


## 2 · Parse each source and mark when each value was available

Every tidy table gets an `available_from` date: the first day the value could have been known. Features may only use
values with `available_from <= as_of`. This is what keeps future data out of the backfill (D7).

| Source | Value | `available_from` |
|---|---|---|
| SAKE | rate per segment and quarter | release date of that quarter (`releases` below) |
| WEA | weekly index, `seco_wwa` only (D2) | week start + `WEA_LAG_DAYS`, at most the run date |
| LIK | monthly inflation `VVP` (D3) | month end + `LIK_LAG_DAYS`, at most the run date |

**About the lags.** Only the current version of each file is available, without past publication dates (D7, limitation).
The lags are therefore set from what was observed in the snapshots and are deliberately on the late side: a value assumed
available too late only makes the backtest slightly pessimistic, while one assumed too early would leak. From now on the pull
log records the real date each new value first appeared, so the lags can be checked after a few weeks of runs. The lags only
matter for the backfill: everything in today's file was available today, so `available_from` is capped at the
run date and the refresh rows always use the newest week and month.

SAKE release dates: the latest quarter of each pulled version uses its real `issued` date from the log. Older quarters use
quarter end + 49 days.

In [6]:
GENDER, NAT, AGE = "GENDER_DE", "NAT_DE", "DETAILS_DE"
NAT_SHORT = {"Schweizer und Schweizerinnen": "Schweizer", "Ausländer and Ausländerinnen": "Ausländer", "Total": "Alle"}


def short_label(gender: str, nat: str, age: str) -> str:
    """Same segment labels as notebook 01."""
    gender = "Alle" if gender == "Total" else gender
    nat = NAT_SHORT.get(nat, nat)
    age = "Alle Alter" if age == "Total" else age.replace(" Jahre", "")
    if gender == nat == "Alle" and age == "Alle Alter":
        return "National total"
    return f"{gender} · {nat} · {age}"


def parse_sake(content: bytes) -> pd.DataFrame:
    raw = pd.read_csv(io.BytesIO(content), dtype=str, encoding="utf-8-sig")
    q = raw[raw["FREQ"] == "Q"].copy()
    q["quarter"] = q["PERIOD"].map(parse_quarter)
    q = q[(q["quarter"] >= START_QUARTER) & (q[AGE] != EXCLUDED_AGE)]
    q["segment"] = [short_label(g, n, a) for g, n, a in zip(q[GENDER], q[NAT], q[AGE])]
    q = q.drop_duplicates(["segment", "quarter"])  # the BFS hierarchy repeats some combinations
    return (
        q.rename(columns={GENDER: "gender", NAT: "nationality", AGE: "age_group", "STATUS": "status"})
        .assign(rate=pd.to_numeric(q["VALUE"], errors="coerce"))
        [["segment", "gender", "nationality", "age_group", "quarter", "rate", "status"]]
        .sort_values(["segment", "quarter"])
        .reset_index(drop=True)
    )


def parse_wea(content: bytes) -> pd.DataFrame:
    wea = pd.read_csv(io.BytesIO(content), parse_dates=["date"])
    wea = wea[wea["structure"] == "seco_wwa"].dropna(subset=["value"])  # 3 structural gaps in week 53 (notebook 01)
    wea = wea[["date", "value"]].sort_values("date").reset_index(drop=True)
    wea["available_from"] = (wea["date"] + pd.Timedelta(days=WEA_LAG_DAYS)).clip(upper=pd.Timestamp(RUN_DATE))
    return wea


def parse_lik(content: bytes) -> pd.DataFrame:
    lik = pd.read_csv(io.BytesIO(content), sep=";", skiprows=2, encoding="utf-8-sig")
    lik = lik[lik["D0"] == "VVP"].dropna(subset=["Value"])
    lik = pd.DataFrame({"month": pd.PeriodIndex(lik["Date"], freq="M"), "value": lik["Value"].astype(float).values})
    lik["available_from"] = (lik["month"].dt.end_time.dt.normalize() + pd.Timedelta(days=LIK_LAG_DAYS)).clip(upper=pd.Timestamp(RUN_DATE))
    return lik.sort_values("month").reset_index(drop=True)


def sake_release_dates(quarters, log: pd.DataFrame) -> pd.Series:
    """Release date per quarter: real 'issued' date where the log has it, quarter end + lag otherwise."""
    releases = pd.Series(
        {q: q.end_time.normalize() + pd.Timedelta(days=SAKE_LAG_DAYS) for q in sorted(set(quarters))}, name="released"
    )
    known = log[(log["source"] == "sake") & log["published"].notna()]
    for latest, published in zip(known["latest_period"], known["published"]):
        q = pd.Period(latest, freq="Q")
        if q in releases.index:
            releases[q] = pd.Timestamp(published)
    return releases


sake = parse_sake(pulls["sake"][0])
wea = parse_wea(pulls["wea"][0])
lik = parse_lik(pulls["lik"][0])
releases = sake_release_dates(sake["quarter"], pull_log)
sake["available_from"] = sake["quarter"].map(releases)

print(f"SAKE: {sake['segment'].nunique()} segments, {sake['quarter'].min()} to {sake['quarter'].max()}, {len(sake):,} rows")
print(f"WEA:  {len(wea):,} weeks, {wea['date'].min().date()} to {wea['date'].max().date()}")
print(f"LIK:  {len(lik):,} months, {lik['month'].min()} to {lik['month'].max()}")
print("latest SAKE release:", releases.index[-1], "on", releases.iloc[-1].date())

SAKE: 45 segments, 2010Q1 to 2026Q2, 2,970 rows
WEA:  1,133 weeks, 2005-01-03 to 2026-10-05
LIK:  1,257 months, 1922-01 to 2026-09
latest SAKE release: 2026Q2 on 2026-08-18


> **Findings (first live run, 25.10.2026)**
>
> - Lags confirmed: LIK for September was published on 21.10, exactly month end + 21 days. The newest WEA week (05.10)
>   was already in the file 20 days after its start, so the assumed 24 days is on the safe side.
> - SECO revised past WEA values between the pulls of 15.10 and 25.10: the 2026Q1 quarterly mean changed from −0.41 to
>   −0.49. This confirms D2 (values can be revised) and is why every pull is stored with its date (D5).

### Validate before building features

The pipeline stops here if a source changed shape: a run that fails loudly is better than a feature table that is
silently wrong. The checks repeat the findings of notebook 01.

In [7]:
def validate_sources(sake: pd.DataFrame, wea: pd.DataFrame, lik: pd.DataFrame) -> None:
    n_quarters = len(pd.period_range(START_QUARTER, sake["quarter"].max(), freq="Q"))
    per_segment = sake.groupby("segment")["quarter"].nunique()

    assert sake["segment"].nunique() == N_SEGMENTS, f"expected {N_SEGMENTS} segments, got {sake['segment'].nunique()}"
    assert (per_segment == n_quarters).all(), f"segments with missing quarters: {per_segment[per_segment < n_quarters]}"
    assert sake["rate"].notna().all(), "SAKE has empty values in scope"
    assert set(sake["status"]) <= {"A", "P", "U"}, f"unexpected STATUS codes: {set(sake['status'])}"
    assert wea["date"].is_unique and (wea["date"].dt.dayofweek == 0).all(), "WEA dates are not unique Mondays"
    assert lik["month"].is_unique, "LIK has duplicate months"
    months = lik.loc[lik["month"] >= START_QUARTER.asfreq("M", "start"), "month"]
    assert len(months) == len(pd.period_range(months.min(), months.max(), freq="M")), "LIK has missing months since 2010"
    print("all source checks passed")


validate_sources(sake, wea, lik)

all source checks passed


## 3 · Feature rows as of a date (D6, D7)

`build_features(as_of)` returns one row per segment. It first finds *t*, the latest SAKE quarter released by `as_of`;
the target is quarter *t+1*. Then it uses only values available on `as_of`:

| Group | Columns | Meaning |
|---|---|---|
| Keys | `as_of`, `kind`, `segment`, `quarter_t`, `target_quarter` | `kind` is `release` (backfill) or `refresh` (run date) |
| Segment | `gender`, `nationality`, `age_group`, `target_qoy` | `target_qoy` = quarter of the year of the target (1–4) |
| Own history | `rate_t`, `rate_t1`, `rate_t3`, `yoy_t`, `status_t` | rate in *t*, *t−1*, *t−3*; `yoy_t` = rate in *t* minus *t−4* |
| WEA | `wea_13w`, `wea_q1` … `wea_q4` | mean of the last 13 available weeks; quarterly means 1–4 quarters before the target |
| Inflation | `lik_latest`, `lik_q1` … `lik_q4` | latest available month; quarterly means 1–4 quarters before the target |
| Audit | `wea_last_week`, `lik_last_month`, `sake_released` | the newest input used, for the leakage check |
| Labels | `y_rate`, `y_yoy` | rate in *t+1*, and its change against *t−3* (D6). Empty for the newest target |

"Quarter 1 before the target" is quarter *t*, "quarter 4" is *t−3*. The labels are filled from the full SAKE table, so
they are **never** used as features: the `y_` prefix is there to make that impossible to miss in MS3.

In [8]:
def available(df: pd.DataFrame, as_of: pd.Timestamp) -> pd.DataFrame:
    return df[df["available_from"] <= as_of]


def quarterly_means(values: pd.Series, quarter_of_value: pd.Series, quarters: list) -> list[float]:
    """Mean of the values in each of the given quarters (NaN if a quarter has no value)."""
    by_quarter = values.groupby(quarter_of_value).mean()
    return [by_quarter.get(q, np.nan) for q in quarters]


def build_features(as_of, sake: pd.DataFrame, wea: pd.DataFrame, lik: pd.DataFrame, kind: str = "release") -> pd.DataFrame:
    as_of = pd.Timestamp(as_of)
    known = available(sake, as_of)
    t = known["quarter"].max()
    target = t + 1
    back = [t, t - 1, t - 2, t - 3]  # 1 to 4 quarters before the target

    rates = known.pivot(index="segment", columns="quarter", values="rate")
    labels = sake[sake["quarter"] == target].set_index("segment")["rate"]  # future values, labels only
    seg = known[known["quarter"] == t].set_index("segment")

    wea_now, lik_now = available(wea, as_of), available(lik, as_of)
    wea_q = quarterly_means(wea_now["value"], wea_now["date"].dt.to_period("Q"), back)
    lik_q = quarterly_means(lik_now["value"], lik_now["month"].dt.asfreq("Q"), back)

    f = pd.DataFrame(
        {
            "as_of": as_of,
            "kind": kind,
            "quarter_t": str(t),
            "target_quarter": str(target),
            "gender": seg["gender"],
            "nationality": seg["nationality"],
            "age_group": seg["age_group"],
            "target_qoy": target.quarter,
            "rate_t": rates[t],
            "rate_t1": rates[t - 1],
            "rate_t3": rates[t - 3],
            "yoy_t": rates[t] - rates[t - 4],
            "status_t": seg["status"],
            "wea_13w": wea_now["value"].tail(13).mean(),
            **{f"wea_q{i}": v for i, v in enumerate(wea_q, start=1)},
            "lik_latest": lik_now["value"].iloc[-1],
            **{f"lik_q{i}": v for i, v in enumerate(lik_q, start=1)},
            "wea_last_week": wea_now["date"].max(),
            "lik_last_month": str(lik_now["month"].max()),
            "sake_released": seg["available_from"],
            "y_rate": labels.reindex(seg.index),
        }
    )
    f["y_yoy"] = f["y_rate"] - f["rate_t3"]
    return f.rename_axis("segment").reset_index()


# One example: the forecast made when Q2 2026 was released
build_features(releases[pd.Period("2026Q2", freq="Q")], sake, wea, lik).head(3).T

,0,1,2
segment,Alle · Alle · 15-24,Alle · Alle · 25-39,Alle · Alle · 40-54
as_of,2026-08-18 00:00:00,2026-08-18 00:00:00,2026-08-18 00:00:00
kind,release,release,release
quarter_t,2026Q2,2026Q2,2026Q2
target_quarter,2026Q3,2026Q3,2026Q3
gender,Total,Total,Total
nationality,Total,Total,Total
age_group,15-24 Jahre,25-39 Jahre,40-54 Jahre
target_qoy,3,3,3
rate_t,7.865038,5.372375,4.49153


## 4 · Backfill and refresh

**Backfill:** `build_features` for every SAKE release from 2011Q1 on (the first quarter with a value 4 quarters earlier, needed
for `yoy_t`). This gives the training set for MS3. Every run recomputes the whole backfill: it takes a few seconds, and it means
a WEA revision or a fix in the code reaches all rows.

**Refresh:** the same function as of the run date. Between SAKE releases this row set changes every week through `wea_13w` and
`lik_latest`, which is what the inference pipeline reads (D1). If the run date is a release date, the refresh would equal the
release rows, so it is skipped.

In [9]:
def backfill(sake, wea, lik, releases, run_date) -> pd.DataFrame:
    first_t = START_QUARTER + 4
    dates = releases[(releases.index >= first_t) & (releases <= pd.Timestamp(run_date))]
    parts = [build_features(d, sake, wea, lik, kind="release") for d in dates]
    if pd.Timestamp(run_date) > dates.max():
        parts.append(build_features(run_date, sake, wea, lik, kind="refresh"))
    return pd.concat(parts, ignore_index=True)


features = backfill(sake, wea, lik, releases, RUN_DATE)
summary = features.groupby("kind").agg(
    rows=("segment", "size"), as_of_dates=("as_of", "nunique"), first=("target_quarter", "min"), last=("target_quarter", "max")
)
summary

,rows,as_of_dates,first,last
kind,,,,
refresh,45,1,2026Q3,2026Q3
release,2790,62,2011Q2,2026Q3


## 5 · Checks: leakage, completeness, signal

Three leakage checks, from weak to strong:

1. **Audit columns.** The newest WEA week, LIK month and SAKE release used by every row are all available on its `as_of`.
2. **Target not yet released.** The release date of every target quarter is after `as_of`, so no row predicts a quarter
   that was already published.
3. **Truncation test.** For a few random dates, delete everything from the sources that was not available yet and rebuild the
   rows. If any feature changes, `build_features` was reading a value it shouldn't have.

Check 3 is the one to point reviewers to. It goes into `tests/` when the code is exported.

In [10]:
FEATURES = [c for c in features.columns if c not in {"as_of", "kind", "segment", "quarter_t", "target_quarter",
                                                       "wea_last_week", "lik_last_month", "sake_released", "y_rate", "y_yoy"}]

# 1 · Audit columns
assert (features["sake_released"] <= features["as_of"]).all()
run_day = pd.Timestamp(RUN_DATE)
assert ((features["wea_last_week"] + pd.Timedelta(days=WEA_LAG_DAYS)).clip(upper=run_day) <= features["as_of"]).all()
lik_avail = pd.PeriodIndex(features["lik_last_month"], freq="M").end_time.normalize() + pd.Timedelta(days=LIK_LAG_DAYS)
assert (lik_avail.to_series().clip(upper=run_day).values <= features["as_of"].values).all()

# 2 · The target quarter was never released before as_of
target_release = pd.PeriodIndex(features["target_quarter"], freq="Q").map(lambda q: releases.get(q, pd.Timestamp.max))
assert (target_release > features["as_of"]).all()

# 3 · Truncation test
rng = np.random.default_rng(0)
for as_of in rng.choice(features["as_of"].unique(), size=5, replace=False):
    as_of = pd.Timestamp(as_of)
    full = features[features["as_of"] == as_of].set_index("segment")[FEATURES]
    cut = build_features(as_of, available(sake, as_of), available(wea, as_of), available(lik, as_of)).set_index("segment")[FEATURES]
    pd.testing.assert_frame_equal(full, cut.loc[full.index], check_dtype=False)
print("leakage checks passed")

# Completeness: no empty features, labels only missing for the newest target
missing = features[FEATURES].isna().sum()
assert missing.sum() == 0, f"empty features: {missing[missing > 0].to_dict()}"
unlabelled = features.loc[features["y_rate"].isna(), "target_quarter"].unique()
print("feature columns:", len(FEATURES), "| rows without label (to forecast):", unlabelled.tolist())

leakage checks passed
feature columns: 19 | rows without label (to forecast): ['2026Q3']


**Signal check.** On the national total, the correlations between the year-on-year target and the WEA and inflation features
should match the exploration (D6: WEA about −0.66 two quarters back, inflation about −0.50). If they don't, the feature code
is shifting something by a quarter.

In [11]:
national = features[(features["segment"] == "National total") & (features["kind"] == "release")].dropna(subset=["y_yoy"])
signal = national[["wea_13w", "wea_q1", "wea_q2", "wea_q3", "wea_q4", "lik_latest", "lik_q1", "lik_q2", "yoy_t"]].corrwith(national["y_yoy"])
signal.round(2).to_frame("corr with y_yoy (national)")

,corr with y_yoy (national)
wea_13w,-0.51
wea_q1,-0.53
wea_q2,-0.64
wea_q3,-0.63
wea_q4,-0.40
lik_latest,-0.51
lik_q1,-0.47
lik_q2,-0.37
yoy_t,0.70


## 6 · Store a new feature version

The feature table is written as one Parquet file per run, next to a manifest:

```
feature_store/features/version=<run date>/features.parquet
feature_store/features/version=<run date>/manifest.json
```

This is the "poor man's feature store" from the course FAQ: versioned Parquet files in a GCS bucket. The manifest records the
hashes of the source files the version was built from, so every forecast can be traced back to its data (D5). The training
pipeline reads the latest version, or a fixed one for a reproducible run. A re-run on the same day overwrites that day's
version.

In [12]:
def store_features(features: pd.DataFrame, run_log: pd.DataFrame) -> str:
    folder = f"{FEATURES_PREFIX}/version={RUN_DATE}"
    buffer = io.BytesIO()
    features.to_parquet(buffer, index=False)
    store.write(f"{folder}/features.parquet", buffer.getvalue())
    manifest = {
        "version": RUN_DATE,
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "offline": OFFLINE,
        "rows": len(features),
        "feature_columns": FEATURES,
        "label_columns": ["y_rate", "y_yoy"],
        "new_sake_release": bool(new_sake_release),
        "lags_days": {"sake": SAKE_LAG_DAYS, "wea": WEA_LAG_DAYS, "lik": LIK_LAG_DAYS},
        "sources": run_log.set_index("source")[["sha256", "published", "latest_period", "download_date"]].to_dict("index"),
    }
    store.write(f"{folder}/manifest.json", json.dumps(manifest, indent=2, default=str).encode())
    return store.uri(folder)


def feature_versions() -> list[str]:
    """All stored versions, oldest first, e.g. ['2026-10-25', '2026-11-02']."""
    keys = store.list(FEATURES_PREFIX)
    return sorted({k.split("version=", 1)[1].split("/", 1)[0] for k in keys if k.endswith("features.parquet")})


def load_features(version: str | None = None) -> pd.DataFrame:
    """Read one feature version, the latest by default."""
    version = version or feature_versions()[-1]
    return pd.read_parquet(io.BytesIO(store.read(f"{FEATURES_PREFIX}/version={version}/features.parquet")))


location = store_features(features, this_run)
reloaded = load_features()
pd.testing.assert_frame_equal(features, reloaded, check_dtype=False)
print("stored", location, f"({len(reloaded):,} rows, read back identical)")
print("versions in the store:", feature_versions())

stored gs://hslu-hs26-mlops-theodora/feature_store/features/version=2026-10-25 (2,835 rows, read back identical)
versions in the store: ['2026-10-25']


## Next

- Export sections 1–4 and 6 to `src/feature_pipeline/`, and the truncation test of section 5 to `tests/`.
- Run the exported pipeline on a schedule with GitHub Actions.